In [1]:
import optax.projections
%load_ext autoreload
%autoreload 2

# XLA_PYTHON_CLIENT_PREALLOCATE = False
import jax
import jax.numpy as jnp
import equinox as eqx

import numpy as np
import matplotlib.pyplot as plt

from qdots_qll.distributions import Distribution, update_particles_locations, update_weights

import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

from jax import jit
from jax.scipy.linalg import expm

from qdots_qll.models.single_dot_weak_coupling_GAME import *

from qdots_qll.resamplers import LWResamplerBounds

from qdots_qll.exp_design import OptimizeInitialStateMeasurements, MaxDetFimExpDesign

from tensorflow_probability.substrates import jax as tfp
import optax


import jax.tree_util as jtu


def tree_stack(trees):
    return jax.tree.map(lambda *v: jnp.stack(v), *trees)

def tree_unstack(tree):
    leaves, treedef = jax.tree.flatten(tree)
    return [treedef.unflatten(leaf) for leaf in zip(*leaves, strict=True)]


def transpose_results(pytree):
    return tree_stack(list(map(list, zip(*tree_unstack(tree_unstack(pytree))))))


In [2]:
# Definition of parameters

boundaries = jnp.array([
    [0.1, 0.5],
    [0.1, 0.5],
    [0.01, 0.2],
    [-0.5, -0.01],
])

seed = 1
no_particles = 1000
no_initial_states = 4
no_measurement_basis = 3

popt = OptimizeInitialStateMeasurements(iter=2, lr=0.01)
expdesign = MaxDetFimExpDesign(t_min=0.01, t_max=45., sgd_iter=4, lr=0.01)
resampler = LWResamplerBounds(a=0.98, parameters_bounds=boundaries)

mus = boundaries.mean(axis=1)
sigmas = jnp.abs((boundaries[:, 0] - boundaries[:, 1]) / (2 * 1))

key = jax.random.PRNGKey(seed=seed)

model = SingleDotWeakCouplingGAME()

key, subkey = jax.random.split(key)
particles_locations = tfp.distributions.TruncatedNormal(loc=mus, scale=sigmas, low=boundaries[:, 0],
                                                        high=boundaries[:, 1]).sample(
    seed=subkey, sample_shape=no_particles)

weights = jnp.ones(no_particles) / no_particles

pdist = Distribution(particles_locations, weights)

p_initial_state = jnp.ones(no_initial_states) / no_initial_states
p_measurement_basis = jnp.ones(no_measurement_basis) / no_measurement_basis

new_p_initial_state, new_p_measurement_basis = p_initial_state, p_measurement_basis

times_list = []
cov_list = []
outcomes_list = []



In [3]:
model

In [8]:
@eqx.filter_jit
def pls_resample(key, distribution):
    resample_result = (resampler.resample)(key, distribution.particles_locations, distribution.weights)
    key, new_weights, new_particles_locations = resample_result['key'], resample_result['weights'], resample_result[
        'particles_locations']
    distribution = Distribution(new_particles_locations, new_weights)
    return key, distribution


@eqx.filter_jit
def do_not_resample(key, distribution):
    return key, distribution


@jax.jit
def select_lkl_outcome(particle, outcome, t, new_p_initial_state, new_p_measurement_basis):
    lkl = model.likelihood_particle_with_basis_initial_state(particle, t, new_p_initial_state, new_p_measurement_basis)[
        *outcome]
    return lkl

In [9]:
# key, subkey = jax.random.split(key)
# 
# t = eqx.filter_jit(expdesign.generate_time)(
#     key=subkey,
#     particles_locations=pdist.particles_locations,
#     weights=pdist.weights,
#     model=model,
#     prob_initial_state=new_p_initial_state,
#     prob_measurement_basis=new_p_measurement_basis,
# )
# times_list.append(t)
# 
# new_dist_initial_state, new_dist_measurement_basis = eqx.filter_jit(
#     popt.optimize_probability_distribution
# )(
#     dist_initial_state=new_p_initial_state,
#     dist_measurement_basis=new_p_measurement_basis,
#     model=model,
#     t=t,
#     particle=pdist.ev(),
# )
# 
# key, subkey = jax.random.split(key)
# chosen_initial_state = jax.random.choice(
#     subkey, jnp.arange(no_initial_states), p=new_dist_initial_state
# )
# 
# key, subkey = jax.random.split(key)
# chosen_basis = jax.random.choice(
#     subkey, jnp.arange(no_measurement_basis), p=new_dist_measurement_basis
# )
# 
# key, subkey = jax.random.split(key)
# outcome = eqx.filter_jit(model.generate_data)(
#     subkey, true_parameters, t, chosen_initial_state, chosen_basis
# )
# outcomes_list.append(outcome)
# 
# lkl_particles = jax.vmap(select_lkl_outcome, in_axes=(0, None))(pdist.particles_locations, outcome)
# 
# pdist = jax.jit(update_weights)(pdist, lkl_particles)
# 
# key, pdist = jax.lax.cond(pdist.check_resampling(), pls_resample, do_not_resample, *(key, pdist))
# 



In [283]:
def initialize_carry(key):
    key, subkey = jax.random.split(key)
    particles_locations = tfp.distributions.TruncatedNormal(loc=mus, scale=sigmas, low=boundaries[:, 0],
                                                            high=boundaries[:, 1]).sample(
        seed=subkey, sample_shape=no_particles)

    weights = jnp.ones(no_particles) / no_particles

    pdist = Distribution(particles_locations, weights)
    return (key, pdist, p_initial_state, p_measurement_basis)


In [419]:
initial_carries = jax.vmap(initialize_carry)(jax.random.split(subkey, 50))

In [10]:
# carry -> (key, pdist, p_initial_state, p_measurement_basis)
# y ->  (outcome, t, pdist)

In [490]:
@jax.jit
def f_scan(carry, _):
    key, pdist, p_initial_state, p_measurement_basis = carry
    key, subkey = jax.random.split(key)

    t = eqx.filter_jit(expdesign.generate_time)(
        key=subkey,
        particles_locations=pdist.particles_locations,
        weights=pdist.weights,
        model=model,
        prob_initial_state=p_initial_state,
        prob_measurement_basis=p_measurement_basis,
    )
    # times_list.append(t)

    p_initial_state, p_measurement_basis = eqx.filter_jit(
        popt.optimize_probability_distribution
    )(
        dist_initial_state=p_initial_state,
        dist_measurement_basis=p_measurement_basis,
        model=model,
        t=t,
        particle=pdist.ev(),
    )

    key, subkey = jax.random.split(key)
    chosen_initial_state = jax.random.choice(
        subkey, jnp.arange(no_initial_states), p=p_initial_state
    )

    key, subkey = jax.random.split(key)
    chosen_basis = jax.random.choice(
        subkey, jnp.arange(no_measurement_basis), p=p_measurement_basis
    )

    key, subkey = jax.random.split(key)
    outcome = eqx.filter_jit(model.generate_data)(
        subkey, true_parameters, t, chosen_initial_state, chosen_basis
    )
    # outcomes_list.append(outcome)

    lkl_particles = jax.vmap(select_lkl_outcome, in_axes=(0, None, None, None, None))(pdist.particles_locations,
                                                                                      outcome, t, p_initial_state,
                                                                                      p_measurement_basis)

    pdist = jax.jit(update_weights)(pdist, lkl_particles)

    key, pdist = jax.lax.cond(pdist.check_resampling(), pls_resample, do_not_resample, *(key, pdist))

    return (key, pdist, p_initial_state, p_measurement_basis), (outcome, t, pdist)


In [383]:
(key, pdist, p_initial_state, p_measurement_basis), _ = f_scan((key, pdist, p_initial_state, p_measurement_basis), _)

In [499]:
(key, pdist, p_initial_state, p_measurement_basis), acumulador =(jax.lax.scan)(f_scan, init=(
    key, pdist, p_initial_state, p_measurement_basis), xs=None, length=100)

In [502]:
f_scan_mapped = (jax.vmap(f_scan, in_axes=(0, None)))

In [517]:
initial_carries = jax.vmap(initialize_carry)(jax.random.split(subkey, 1000))

In [540]:
from time import process_time

key, subkey = jax.random.split(key)
no_keys = 100

initial_carries = jax.vmap(initialize_carry)(jax.random.split(subkey, no_keys))

t1_start = process_time()
(jax.lax.scan)(f_scan_mapped, init=initial_carries, xs=None, length=1)
t1_stop = process_time()

comp_time = t1_stop - t1_start


t1_start = process_time()
(jax.lax.scan)(f_scan_mapped, init=initial_carries, xs=None, length=1)
t1_stop = process_time()

iter_time = t1_stop - t1_start

print(comp_time)
print(iter_time)



In [529]:
f_scan(initial_carries[0], None)

In [584]:
_, results = (jax.lax.scan)(f_scan_mapped, init=initial_carries, xs=None, length=10)

In [585]:
outcomes = results[0]
times = results[1]
dists = results[2]


In [661]:
import joblib

joblib.dump(results, "results.job")

In [662]:
ej = joblib.load("results.job")

In [663]:
ej

In [551]:
from datetime import timedelta
def estimate_run_time(no_keys, no_iters):
    secs =  15 + 0.5/100*no_keys*no_iters
    return str(timedelta(seconds=secs))


estimate_run_time(100, 5000)

In [439]:
for i in range(100):
    carries, _ = (jax.vmap(f_scan, in_axes=(0, None)))(carries, _)
    if i==0:
        print("first iteration")
    

In [268]:
for _ in range(1000):
    print(key)
    (key, pdist, p_initial_state, p_measurement_basis), acumulador = f_scan(
        (key, pdist, p_initial_state, p_measurement_basis), _)
    outcomes_list.append(acumulador[0])
    times_list.append(acumulador[1])

